# Huấn luyện DUSN-X state updater và router
Không fine-tune LLM. Chạy từng ô theo thứ tự. Notebook chưa được chạy trên Colab bởi người duy trì. Nhãn synthetic và holdout chưa được duyệt độc lập.

In [ ]:
import sys, subprocess, pathlib, os
assert (3, 11) <= sys.version_info[:2] < (3, 14), "Cần Python 3.11–3.13"
PROJECT = pathlib.Path("/content/dusnx-platform")
REPOSITORY = "https://github.com/quocdoanhdang2-sketch/dusnx-platform.git"
REVISION = "main"  # Đổi thành commit đã duyệt để tái lập chính xác.
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(PROJECT)], check=True)
    subprocess.run(["git", "checkout", REVISION], cwd=PROJECT, check=True)
os.chdir(PROJECT)
print("Repository HEAD:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "./python[dev,data]"], check=True)
sys.path[:0] = [str(PROJECT / "python/src"), str(PROJECT / "python"), str(PROJECT / "python/scripts")]
os.environ["PYTHONPATH"] = os.pathsep.join(sys.path[:3])


In [ ]:
from colab_helper import check_environment
SMOKE = True  # Đổi False chỉ sau khi kiểm tra dữ liệu và chọn GPU.
print(check_environment(large=not SMOKE))


## Lưu vào Drive
Bấm Connect to Google Drive rồi chọn tài khoản và cấp quyền. Không đặt token vào ô code hoặc output. Mỗi lần chạy mới dùng RUN_NAME khác; resume dùng lại tên cũ.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUN_NAME = "dusnx-router-v1"
OUTPUT = pathlib.Path("/content/drive/MyDrive/DUSNX") / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Smoke không cần mạng/dataset ngoài. Train lớn tải nguồn public theo manifest cố định.
if not SMOKE:
    subprocess.run([sys.executable, "python/scripts/fetch_sources.py", "--source", "all"], check=True)
    subprocess.run([sys.executable, "python/scripts/import_data.py", "--source", "massive",
                    "--input", "runtime/external-data/massive/vi-VN.jsonl",
                    "--output-dir", "runtime/imported/massive"], check=True)
# CSConDa mặc định bỏ qua. Chỉ sau khi HF chấp thuận, dùng Colab Secrets HF_TOKEN
# và gọi fetch_sources.py --source csconda --allow-gated. Không in token.


In [ ]:
# Legacy 30k/checkpoint không nằm trong git: tải bản local của bạn lên thư mục Drive này.
LEGACY = pathlib.Path("/content/drive/MyDrive/DUSNX/input/synthetic_30k_v2.jsonl")
BASELINE = pathlib.Path("/content/drive/MyDrive/DUSNX/input/dusnx_smoke_v2.pt")
from prepare_data import prepare
external = PROJECT / "runtime/imported/massive/candidates.jsonl"
report = prepare(PROJECT / "runtime/prepared", legacy=LEGACY if LEGACY.exists() else None,
                 external=external if external.exists() and not SMOKE else None)
import shutil
shutil.copy2(PROJECT / "runtime/prepared/manifest.json", OUTPUT / "data_manifest.json")
shutil.copy2(PROJECT / "configs/data_sources.json", OUTPUT / "data_sources.json")
shutil.copy2(PROJECT / "configs/massive_vi_mapping.json", OUTPUT / "massive_vi_mapping.json")
print(report)
print("MASSIVE chờ mapping thì không có nhãn train; agent/action không được suy diễn.")


In [ ]:
# CPU smoke kiểm tra train+resume bằng kiến trúc nhỏ; không phải kết quả chất lượng.
if SMOKE:
    from pipeline_smoke import smoke
    smoke(OUTPUT / "smoke")
    CHECKPOINT = OUTPUT / "smoke/smoke.pt"
else:
    from colab_helper import drive_config
    from dusnx_core.training import train
    from colab_helper import BASELINE_SHA256
    from dusnx_core.data_pipeline import sha256
    if BASELINE.exists():
        assert sha256(BASELINE) == BASELINE_SHA256, "Sai checkpoint mốc 23aaa48"
    config = drive_config(PROJECT, OUTPUT)
    RESUME = False  # True nếu có router.last.pt từ lần chạy trước cùng config/data.
    train(config, resume=OUTPUT / "router.last.pt" if RESUME else None,
          init_from=BASELINE if BASELINE.exists() and not RESUME else None)
    CHECKPOINT = OUTPUT / "router.pt"


## Đánh giá sau khi nhãn đã khóa
Mock chỉ kiểm tra pipeline và routing; không chứng minh chất lượng câu trả lời LLM. Để đo đầy đủ, chạy lại ở local có Ollama với provider=ollama. Không sửa gold sau khi xem prediction.

In [ ]:
from colab_helper import evaluate_pair
print(evaluate_pair(PROJECT, CHECKPOINT, BASELINE, OUTPUT / "evaluation", provider="mock"))


In [ ]:
import shutil
destination = PROJECT / "artifacts" / RUN_NAME
shutil.copytree(OUTPUT, destination, dirs_exist_ok=True)
print("Đã lưu checkpoint, config, seed, hash data, nhãn và metric epoch tại", OUTPUT)
print("Windows: mở Google Drive, bấm phải thư mục run → Download; giải nén vào artifacts/.")
